# 01 · Limpieza de Datos y Análisis Exploratorio (EDA)
> **Canal:** corporativo  
> **Pipeline:** Carga → Fusión → Homologación → Geografía → Features → EDA → Modelo → Exportación

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import unicodedata
from sklearn.preprocessing import StandardScaler

In [ ]:
# ── Paleta y estilo corporativo ────────────────────────────────
C_VERDE    = '#24743c'
C_V_PAL    = '#79a65c'
C_AMARILLO = '#f39c14'
C_A_PAL    = '#f3eb8b'
PALETTE    = [C_VERDE, C_V_PAL, C_AMARILLO, C_A_PAL]

plt.rcParams.update({
    'figure.facecolor'  : 'white',
    'axes.facecolor'    : '#f5fbf7',
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'axes.spines.left'  : True,
    'axes.spines.bottom': True,
    'axes.edgecolor'    : '#b0c8b8',
    'axes.grid'         : True,
    'axes.grid.axis'    : 'y',
    'grid.alpha'        : 0.4,
    'grid.color'        : '#c0d8c8',
    'font.size'         : 11,
    'axes.titlesize'    : 13,
    'axes.titleweight'  : 'bold',
    'axes.titlecolor'   : C_VERDE,
    'axes.labelcolor'   : '#333333',
    'xtick.color'       : '#444444',
    'ytick.color'       : '#444444',
})
print('Paleta corporativa cargada.')

## 1. Carga de datos

In [ ]:
PAP_clientes = pd.read_csv('PAP_informacion_clientes.csv')
PAP_clientes['NIT'] = PAP_clientes['NIT'].astype(str).str.replace('^C', '', regex=True).str.strip()
Productos_homologacion = pd.read_excel('Productos.xlsx')
segmentacion_raw       = pd.read_excel('Segmentacion Base JUL25.xlsx', sheet_name='BASE JULIO-25')
df_acum                = pd.read_csv('ReporteAcum.csv', sep=';', encoding='utf-8-sig')

print(f'Clientes:     {PAP_clientes.shape}')
print(f'Segmentacion: {segmentacion_raw.shape}')
print(f'Acumulado:    {df_acum.shape}')

In [ ]:
PAP_clientes.columns

## 2. Limpieza y fusión de fuentes

In [ ]:
def limpiar_nombre(col):
    col = col.upper().replace('\n', '_').replace(' ', '_')
    for s, d in [('\u00c1','A'),('\u00c9','E'),('\u00cd','I'),('\u00d3','O'),('\u00da','U')]:
        col = col.replace(s, d)
    col = re.sub(r'[^A-Z0-9_]', '', col)
    col = re.sub(r'_+', '_', col)
    return col.strip('_')

COLS_SEG = [
    'NIT', 'CONTRATANTE', 'ANTIGEDAD', 'RANGO_ANTIGEDAD', 'RANGO_AFILIADOS',
    'RANGO_EDAD', 'RANGO_TARIFA', 'RANGO_PRODUCCION', 'RANGO_MORTALIDAD',
    'RANGO_SINIESTRALIDAD', 'RENTABILIDAD', 'TOP_RESULTADOS', 'TIPO_ENTIDAD_1', 'TIPO_ENTIDAD_2'
]
segmentacion_raw.columns = [limpiar_nombre(c) for c in segmentacion_raw.columns]
segmentacion = segmentacion_raw[COLS_SEG].copy()
segmentacion['NIT'] = segmentacion['NIT'].astype(str).str.strip()

In [ ]:
PAP_clientes['NIT'] = PAP_clientes['NIT'].astype(str).str.strip()
df_activos = PAP_clientes.merge(segmentacion, on='NIT', how='left')
assert len(df_activos) == len(PAP_clientes), 'El merge alteró el número de registros'
print(f'PAP: {len(PAP_clientes):,}  →  unido segmentación: {len(df_activos):,}')

In [ ]:
# Separar combinacion, excluir adicionales y normalizar clave
_split = df_acum['Combinacion'].astype(str).str.split('|', n=1, expand=True)
df_acum['Combinacion']   = _split[0]
df_acum['Combinacion_2'] = _split[1] if 1 in _split.columns else None

df_acum = df_acum[df_acum['Tipo_Producto'] != 'ADICIONALES'].copy()
df_acum['Combinacion'] = (
    df_acum['Combinacion'].astype(str).str.extract(r'(\d+)', expand=False).str.strip()
)

# Solo columnas necesarias para evitar conflictos de nombres en el merge
df_acum_join = df_acum[['Combinacion', 'Canal', 'Act.Valor']].drop_duplicates(subset=['Combinacion'])
print(f'Acumulado listo: {df_acum_join.shape}, duplicados clave: {df_acum_join["Combinacion"].duplicated().sum()}')

In [ ]:
df_activos['Contrato'] = (
    df_activos['Contrato'].astype(str).str.extract(r'(\d+)', expand=False).str.strip()
)

# Conservar el registro mas completo por contrato duplicado
n_dup = df_activos['Contrato'].duplicated().sum()
print(f'Duplicados en activos: {n_dup}')
if n_dup > 0:
    df_activos = (
        df_activos
        .assign(
            _campos_llenos=df_activos.notna().sum(axis=1),
            _dir_len=df_activos['Direccion'].fillna('').str.len()
        )
        .sort_values(['_campos_llenos', '_dir_len'], ascending=False)
        .drop_duplicates(subset='Contrato', keep='first')
        .drop(columns=['_campos_llenos', '_dir_len'])
        .reset_index(drop=True)
    )
    print(f'  -> duplicados eliminados. Shape: {df_activos.shape}')

df_final = df_activos.merge(
    df_acum_join,
    left_on='Contrato', right_on='Combinacion', how='left'
).drop(columns=['Combinacion'])

# Capitalizar y resolver posibles columnas duplicadas
df_final.columns = df_final.columns.str.capitalize()
df_final = df_final.loc[:, ~df_final.columns.duplicated(keep='first')]
print(f'df_final: {df_final.shape}')

## 3. Homologación de productos y filtro de canales

In [ ]:
MAPEO_ADICIONAL = {
    'INTEGRAL - EXTRA'             : 'INTEGRAL',
    'PLAN INTEGRAL CONTINUIDAD'    : 'INTEGRAL',
    'CANDELARIA DIEZ + RP'         : 'DIEZ',
    'CANDELARIA ADICIONAL MAYOR 65': 'CANDELARIA',
}

# Evitar Producto_x / Producto_y si PAP ya trae columna Producto
if 'Producto' in df_final.columns:
    df_final = df_final.drop(columns=['Producto'])

df_final = (
    df_final.merge(
        Productos_homologacion[['Nombre_plan', 'Producto']].rename(columns={'Nombre_plan': '_key_plan'}),
        left_on='Nombreproducto', right_on='_key_plan', how='left'
    ).drop(columns=['_key_plan'])
)

sin_plan = df_final['Producto'].isna()
df_final.loc[sin_plan, 'Producto'] = df_final.loc[sin_plan, 'Nombreproducto'].map(MAPEO_ADICIONAL)
n_sin_plan = df_final['Producto'].isna().sum()
if n_sin_plan > 0:
    print(f'Sin plan: {n_sin_plan} — Nombreproducto sin mapeo:')
    print(df_final.loc[df_final['Producto'].isna(), 'Nombreproducto'].value_counts().to_string())
    df_final['Producto'] = df_final['Producto'].fillna('Sin_plan')
else:
    print('OK - Todos los registros tienen plan asignado.')

# Complemento de individuales: strip todos los prefijos, excluir canales individuales
CANALES_INDIVIDUALES = {
    'PAP - BOGOTA', 'PAP - BOYACA', 'PAP - SOACHA',
    'INDIVIDUALES - BOGOTA', 'INDIVIDUALES - SOACHA',
    'CONTACT CENTER', 'MASIVO','MASIVO - BOGOTA','WEB - BOGOTA',
}
df_final['Canal'] = df_final['Canal'].str.replace(r'^(IND|GXR|GR) - ', '', regex=True)
df_final = df_final[~df_final['Canal'].isin(CANALES_INDIVIDUALES)].copy()

# Excluir registros sin canal asignado (no trazables a un canal corporativo)
antes = len(df_final)
df_final = df_final[df_final['Canal'].notna()].copy()
print(f'Sin canal eliminados : {antes - len(df_final):,}')
print(f'Registros corporativos: {df_final.shape[0]:,}')
print(df_final['Canal'].value_counts().to_string())

## 4. Procesamiento geográfico (Ciudad → Departamento → Región)

In [ ]:
def normalizar_texto(texto):
    """Mayusculas, sin tildes, sin especiales. Retorna None si es numerico o nulo."""
    if pd.isna(texto):
        return None
    texto = str(texto).strip().upper()
    if texto.replace(' ', '').isdigit():
        return None
    texto = ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )
    return re.sub(r'\s+', ' ', texto).strip()

def capitalizar_ciudad(ciudad):
    """Capitaliza respetando preposiciones (de, del, la, ...)."""
    if not ciudad or pd.isna(ciudad):
        return None
    minusculas = {'de', 'del', 'la', 'las', 'los', 'y', 'el'}
    palabras = ciudad.lower().split()
    return ' '.join(
        p.capitalize() if i == 0 or p not in minusculas else p
        for i, p in enumerate(palabras)
    )

In [ ]:
MAP_CIUDADES = {
    'LA UNION (ANTIOQUIA)': 'La Union', 'LA UNION (VALLE DEL CAUCA)': 'La Union',
    'SAN ANDRES DE TUMACO': 'Tumaco', 'CIENEGA (BOYACA)': 'Cienega',
    'SANTA BARBARA (ANTIOQUIA)': 'Santa Barbara', 'SAN LUIS (ANTIOQUIA)': 'San Luis',
    'BOLIVAR (CAUCA)': 'Bolivar', 'CALARCA': 'Calarca',
    'SAN VICENTE DEL CAGUAN': 'San Vicente del Caguan', 'COLON (PUTUMAYO)': 'Colon',
    'RIOHACHA': 'Riohacha', 'QUIBDO': 'Quibdo', 'LA VIRGINIA': 'La Virginia',
    'MONTELIBANO': 'Montelibano', 'SAN ANDRES': 'San Andres',
    'PUERTO COLOMBIA (ATLANTICO)': 'Puerto Colombia',
    'SANTA FE DE ANTIOQUIA': 'Santa Fe de Antioquia',
    'SANTA ROSA (BOLIVAR)': 'Santa Rosa', 'VILLANUEVA (CASANARE)': 'Villanueva',
    'SANTA MARIA (BOYACA)': 'Santa Maria', 'GRANADA (META)': 'Granada',
    'GRANADA (CUNDINAMARCA)': 'Granada', 'RIONEGRO (SANTANDER)': 'Rionegro',
    'SUAREZ (TOLIMA)': 'Suarez', 'ARGELIA (CAUCA)': 'Argelia',
    'SAN MARTIN (META)': 'San Martin', 'SAN MARTIN (CESAR)': 'San Martin',
    'PUERTO LIBERTADOR': 'Puerto Libertador', 'PAEZ (CAUCA)': 'Paez',
    'VALPARAISO (CAQUETA)': 'Valparaiso', 'GUADALUPE (HUILA)': 'Guadalupe',
    'SAN PEDRO (ANTIOQUIA)': 'San Pedro', 'SAN PEDRO (VALLE DEL CAUCA)': 'San Pedro',
    'BETULIA (ANTIOQUIA)': 'Betulia', 'BETULIA (SANTANDER)': 'Betulia',
    'ALBAN (CUNDINAMARCA)': 'Alban', 'ALBANIA (LA GUAJIRA)': 'Albania',
    'MOSQUERA (CUNDINAMARCA)': 'Mosquera', 'FLORENCIA (CAQUETA)': 'Florencia',
    'ARMENIA (QUINDIO)': 'Armenia', 'RIONEGRO (ANTIOQUIA)': 'Rionegro',
    'CIENAGA (MAGDALENA)': 'Cienaga', 'LA VEGA (CUNDINAMARCA)': 'La Vega',
    'BARBOSA (SANTANDER)': 'Barbosa', 'RIOSUCIO (CALDAS)': 'Riosucio',
    'CANDELARIA (VALLE DEL CAUCA)': 'Candelaria',
    'SAN FRANCISCO (ANTIOQUIA)': 'San Francisco', 'RESTREPO (META)': 'Restrepo',
    'CALDAS (ANTIOQUIA)': 'Caldas', 'CALDAS (BOYACA)': 'Caldas',
    'VILLANUEVA (LA GUAJIRA)': 'Villanueva',
    'PUERTO COLOMBIA (GUAINIA)': 'Puerto Colombia',
    'BELEN (BOYACA)': 'Belen', 'LA PAZ (CESAR)': 'La Paz',
    'SUCRE (SUCRE)': 'Sucre', 'CONCEPCION (ANTIOQUIA)': 'Concepcion',
    'PALESTINA (CALDAS)': 'Palestina'
}

In [ ]:
DEPARTAMENTOS_CIUDADES = {
    'Amazonas': ['Leticia', 'Puerto Narino'],
    'Antioquia': [
        'Abejorral','Abriaqui','Alejandria','Amaga','Amalfi','Andes','Angelopolis','Angostura',
        'Anori','Anza','Apartado','Arboletes','Argelia','Armenia','Barbosa','Bello','Belmira',
        'Betania','Betulia','Briceno','Buritica','Caceres','Caicedo','Caldas','Campamento',
        'Canasgordas','Caracoli','Caramanta','Carepa','Carolina del Principe','Caucasia',
        'Chigorodo','Cisneros','Ciudad Bolivar','Cocorna','Concepcion','Concordia','Copacabana',
        'Dabeiba','Donmatias','Ebejico','El Bagre','El Carmen de Viboral','El Penol','El Retiro',
        'El Santuario','Entrerrios','Envigado','Fredonia','Frontino','Giraldo','Girardota',
        'Gomez Plata','Granada','Guadalupe','Guarne','Guatape','Heliconia','Hispania','Itagui',
        'Ituango','Jardin','Jerico','La Ceja','La Estrella','La Pintada','La Union','Liborina',
        'Maceo','Marinilla','Medellin','Montebello','Murindo','Mutata','Narino','Nechi',
        'Necocli','Olaya','Peque','Pueblorrico','Puerto Berrio','Puerto Nare','Puerto Triunfo',
        'Remedios','Rionegro','Sabanalarga','Sabaneta','Salgar','San Andres de Cuerquia',
        'San Carlos','San Francisco','San Jeronimo','San Jose de la Montana','San Juan de Uraba',
        'San Luis','San Pedro','San Pedro de los Milagros','San Rafael','San Roque','San Vicente',
        'Santa Barbara','Santa Fe de Antioquia','Santa Rosa de Osos','Santo Domingo','Segovia',
        'Sonson','Sopetran','Tamesis','Taraza','Tarso','Titiribi','Toledo','Turbo','Uramita',
        'Urrao','Valdivia','Valparaiso','Vegachi','Venecia','Vigia del Fuerte','Yali','Yarumal',
        'Yolombo','Yondo','Zaragoza'
    ],
    'Arauca': ['Arauca','Arauquita','Cravo Norte','Fortul','Puerto Rondon','Saravena','Tame'],
    'Atlantico': [
        'Baranoa','Barranquilla','Campo de la Cruz','Candelaria','Galapa','Juan de Acosta',
        'Luruaco','Malambo','Manati','Palmar de Varela','Piojo','Polonuevo','Ponedera',
        'Puerto Colombia','Repelon','Sabanagrande','Sabanalarga','Santa Lucia','Santo Tomas',
        'Soledad','Suan','Tubara','Usiacuri'
    ],
    'Bolivar': [
        'Achi','Altos del Rosario','Arenal','Arjona','Arroyohondo','Barranco de Loba','Calamar',
        'Cantagallo','Cartagena','Cicuco','Clemencia','Cordoba','El Carmen de Bolivar','El Guamo',
        'Hatillo de Loba','Magangue','Mahates','Margarita','Maria la Baja','Montecristo','Morales',
        'Mompos','Norosi','Pinillos','Regidor','Rio Viejo','San Cristobal','San Estanislao',
        'San Fernando','San Jacinto','San Juan Nepomuceno','San Martin de Loba','San Pablo',
        'Santa Catalina','Santa Rosa','Simiti','Soplaviento','Talaigua Nuevo','Tiquisio',
        'Turbaco','Turbana','Villanueva','Zambrano'
    ],
    'Boyaca': [
        'Almeida','Aquitania','Arcabuco','Belen','Berbeo','Beteitiva','Boavita','Boyaca','Briceno',
        'Buenavista','Busbanza','Caldas','Campohermoso','Cerinza','Chinavita','Chiquinquira',
        'Chiquiza','Chiscas','Chita','Chitaraque','Chivata','Chivor','Cienega','Combita','Coper',
        'Corrales','Covarachia','Cubara','Cucaita','Cuitiva','Duitama','El Cocuy','El Espino',
        'Firavitoba','Floresta','Gachantiva','Gameza','Garagoa','Guacamayas','Guateque','Guayata',
        'Guican','Iza','Jenesano','Jerico','La Capilla','La Uvita','La Victoria','Labranzagrande',
        'Macanal','Maripo','Miraflores','Mongua','Mongui','Moniquira','Motavita','Muzo','Nobsa',
        'Nuevo Colon','Oicata','Otanche','Pachavita','Paez','Paipa','Pajarito','Panqueba','Pauna',
        'Paya','Paz del Rio','Pesca','Pisba','Puerto Boyaca','Quipama','Ramiriqui','Raquira',
        'Rondon','Saboya','Sachica','Samaca','San Eduardo','San Jose de Pare','San Luis de Gaceno',
        'San Mateo','San Miguel de Sema','San Pablo de Borbur','Santa Maria','Santa Rosa de Viterbo',
        'Santa Sofia','Santana','Sativanorte','Sativasur','Siachoque','Soata','Socha','Socota',
        'Sogamoso','Somondoco','Sora','Soraca','Sotaquira','Susacon','Sutamarcha','Sutatenza',
        'Tasco','Tenza','Tibana','Tibasosa','Tinjaca','Tipacoque','Toca','Togui','Topaga','Tota',
        'Tunja','Tunungua','Turmeque','Tuta','Tutaza','Umbita','Ventaquemada','Villa de Leyva',
        'Viracacha','Zetaquira'
    ],
    'Caldas': [
        'Aguadas','Anserma','Aranzazu','Belalcazar','Chinchina','Filadelfia','La Dorada',
        'La Merced','Manizales','Manzanares','Marmato','Marquetalia','Marulanda','Neira',
        'Norcasia','Pacora','Palestina','Pensilvania','Riosucio','Risaralda','Salamina',
        'Samana','San Jose','Supia','Victoria','Villamaria','Viterbo'
    ],
    'Caqueta': [
        'Albania','Belen de los Andaquies','Cartagena del Chaira','Curillo','El Doncello',
        'El Paujil','Florencia','La Montanita','Milan','Morelia','Puerto Rico',
        'San Jose del Fragua','San Vicente del Caguan','Solano','Solita','Valparaiso'
    ],
    'Cauca': [
        'Almaguer','Argelia','Balboa','Bolivar','Buenos Aires','Cajibio','Caldono','Caloto',
        'Corinto','El Tambo','Florencia','Guachene','Guapi','Inza','Jambalo','La Sierra',
        'La Vega','Lopez de Micay','Mercaderes','Miranda','Morales','Padilla','Paez','Patia',
        'Piamonte','Piendamo','Popayan','Puerto Tejada','Purace','Rosas','San Sebastian',
        'Santa Rosa','Santander de Quilichao','Silvia','Sotara','Suarez','Sucre','Timbio',
        'Timbiqui','Toribio','Totoro','Villa Rica'
    ],
    'Cesar': [
        'Aguachica','Agustin Codazzi','Astrea','Becerril','Bosconia','Chimichagua','Chiriguana',
        'Curumani','El Copey','El Paso','Gamarra','Gonzalez','La Gloria','La Jagua de Ibirico',
        'La Paz','Manaure Balcon del Cesar','Pailitas','Pelaya','Pueblo Bello','Rio de Oro',
        'San Alberto','San Diego','San Martin','Tamalameque','Valledupar'
    ],
    'Choco': [
        'Acandi','Alto Baudo','Bagado','Bahia Solano','Bajo Baudo','Bojaya',
        'Canton de San Pablo','Certegui','Condoto','El Atrato','El Carmen de Atrato',
        'El Carmen del Darien','Istmina','Jurado','Litoral de San Juan','Lloro',
        'Medio Atrato','Medio Baudo','Medio San Juan','Novita','Nuqui','Quibdo',
        'Rio Iro','Rio Quito','Riosucio','San Jose del Palmar','Sipi','Tado',
        'Union Panamericana','Unguia'
    ],
    'Cundinamarca': [
        'Agua de Dios','Alban','Anapoima','Anolaima','Apulo','Arbelaez','Beltran','Bituima',
        'Bogota','Bojaca','Cabrera','Cachipay','Cajica','Caparrapi','Caqueza','Carmen de Carupa',
        'Chaguani','Chia','Chipaque','Choachi','Choconta','Cogua','Cota','Cucunuba','El Colegio',
        'El Penon','El Rosal','Facatativa','Fomeque','Fosca','Funza','Fuquene','Fusagasuga',
        'Gachala','Gachancipa','Gacheta','Gama','Girardot','Granada','Guacheta','Guaduas',
        'Guasca','Guataqui','Guatavita','Guayabal de Siquima','Guayabetal','Gutierrez',
        'Jerusalem','Junin','La Calera','La Mesa','La Palma','La Pena','La Vega','Lenguazaque',
        'Macheta','Madrid','Manta','Medina','Mosquera','Narino','Nemocon','Nilo','Nimaima',
        'Nocaima','Pacho','Paime','Pandi','Paratebueno','Pasca','Puerto Salgar','Puli',
        'Quebradanegra','Quetame','Quipile','Ricaurte','San Antonio del Tequendama',
        'San Bernardo','San Cayetano','San Francisco','San Juan de Rioseco','Sasaima',
        'Sesquile','Sibate','Silvania','Simijaca','Soacha','Sopo','Subachoque','Suesca',
        'Supata','Susa','Sutatausa','Tabio','Tausa','Tena','Tenjo','Tibacuy','Tibirita',
        'Tocaima','Tocancipa','Topaipo','Ubala','Ubaque','Ubate','Une','Utica','Venecia',
        'Vergara','Viani','Villagomez','Villapinzon','Villeta','Viota','Yacopi','Zipacon','Zipaquira'
    ],
    'Cordoba': [
        'Ayapel','Buenavista','Canalete','Cerete','Chima','Chinu','Cienaga de Oro','Cotorra',
        'La Apartada','Lorica','Los Cordobas','Momil','Montelibano','Monteria','Monitos',
        'Planeta Rica','Pueblo Nuevo','Puerto Escondido','Puerto Libertador','Purisima',
        'Sahagun','San Andres de Sotavento','San Antero','San Bernardo del Viento','San Carlos',
        'San Jose de Ure','San Pelayo','Tierralta','Tuchin','Valencia'
    ],
    'Guainia': ['Inirida'],
    'Guaviare': ['Calamar','El Retorno','Miraflores','San Jose del Guaviare'],
    'Huila': [
        'Acevedo','Agrado','Aipe','Algeciras','Altamira','Baraya','Campoalegre','Colombia',
        'El Pital','Elias','Garzon','Gigante','Guadalupe','Hobo','Iquira','Isnos',
        'La Argentina','La Plata','Nataga','Neiva','Oporapa','Paicol','Palermo','Palestina',
        'Pitalito','Rivera','Saladoblanco','San Agustin','Santa Maria','Suaza','Tarqui',
        'Tello','Teruel','Tesalia','Timana','Villavieja','Yaguara'
    ],
    'La Guajira': [
        'Albania','Barrancas','Dibulla','Distraccion','El Molino','Fonseca','Hatonuevo',
        'La Jagua del Pilar','Maicao','Manaure','Riohacha','San Juan del Cesar',
        'Uribia','Urumita','Villanueva'
    ],
    'Magdalena': [
        'Algarrobo','Aracataca','Ariguani','Cerro de San Antonio','Chibolo','Cienaga',
        'Concordia','El Banco','El Pinon','El Reten','Fundacion','Guamal','Nueva Granada',
        'Pedraza','Pijino del Carmen','Pivijay','Plato','Pueblo Viejo','Remolino',
        'Sabanas de San Angel','Salamina','San Sebastian de Buenavista','San Zenon',
        'Santa Ana','Santa Barbara de Pinto','Santa Marta','Sitionuevo','Tenerife',
        'Zapayan','Zona Bananera'
    ],
    'Meta': [
        'Acacias','Barranca de Upia','Cabuyaro','Castilla la Nueva','Cubarral','Cumaral',
        'El Calvario','El Castillo','El Dorado','Fuente de Oro','Granada','Guamal',
        'La Macarena','La Uribe','Lejanias','Maripan','Mesetas','Puerto Concordia',
        'Puerto Gaitan','Puerto Lleras','Puerto Lopez','Puerto Rico','Restrepo',
        'San Carlos de Guaroa','San Juan de Arama','San Juanito','San Martin',
        'Villavicencio','Vista Hermosa'
    ],
    'Narino': [
        'Aldana','Ancuya','Arboleda','Barbacoas','Belen','Buesaco','Chachagui','Colon',
        'Consaca','Contadero','Cordoba','Cuaspud','Cumbal','Cumbitara','El Charco','El Penol',
        'El Rosario','El Tablon','El Tambo','Francisco Pizarro','Funes','Guachucal',
        'Guaitarilla','Gualmatan','Iles','Imues','Ipiales','La Cruz','La Florida','La Llanada',
        'La Tola','La Union','Leiva','Linares','Los Andes','Magui Payan','Mallama','Mosquera',
        'Narino','Olaya Herrera','Ospina','Pasto','Policarpa','Potosi','Providencia','Puerres',
        'Pupiales','Ricaurte','Roberto Payan','Samaniego','San Bernardo','San Jose de Alban',
        'San Lorenzo','San Pablo','San Pedro de Cartago','Sandona','Santa Barbara','Santacruz',
        'Sapuyes','Taminango','Tangua','Tumaco','Tuquerres','Yacuanquer'
    ],
    'Norte de Santander': [
        'Abrego','Arboledas','Bochalema','Bucarasica','Cachira','Cacota','Chinacota','Chitaga',
        'Convencion','Cucuta','Cucutilla','Durania','El Carmen','El Tarra','El Zulia',
        'Gramalote','Hacari','Herran','La Esperanza','La Playa de Belen','Labateca','Los Patios',
        'Lourdes','Mutiscua','Ocana','Pamplona','Pamplonita','Puerto Santander','Ragonvalia',
        'Salazar de Las Palmas','San Calixto','San Cayetano','Santiago','Santo Domingo de Silos',
        'Sardinata','Teorama','Tibu','Toledo','Villa Caro','Villa del Rosario'
    ],
    'Putumayo': [
        'Colon','Mocoa','Orito','Puerto Asis','Puerto Caicedo','Puerto Guzman',
        'Puerto Leguizamo','San Francisco','San Miguel','Santiago','Sibundoy',
        'Valle del Guamuez','Villagarzon'
    ],
    'Quindio': [
        'Armenia','Buenavista','Calarca','Circasia','Cordoba','Filandia','Genova',
        'La Tebaida','Montenegro','Pijao','Quimbaya','Salento'
    ],
    'Risaralda': [
        'Apia','Balboa','Belen de Umbria','Dosquebradas','Guatica','La Celia','La Virginia',
        'Marsella','Mistrato','Pereira','Pueblo Rico','Quinchia','Santa Rosa de Cabal','Santuario'
    ],
    'San Andres': ['Providencia y Santa Catalina Islas', 'San Andres'],
    'Santander': [
        'Aguada','Albania','Aratoca','Barbosa','Barichara','Barrancabermeja','Betulia','Bolivar',
        'Bucaramanga','Cabrera','California','Capitanejo','Carcasi','Cepita','Cerrito','Charala',
        'Charta','Chima','Chipata','Cimitarra','Concepcion','Confines','Contratacion','Coromoro',
        'Curiti','El Carmen de Chucuri','El Guacamayo','El Penol','El Playon','El Socorro',
        'Encino','Enciso','Florian','Floridablanca','Galan','Gambita','Giron','Guaca','Guadalupe',
        'Guapota','Guavata','Guepsa','Hato','Jesus Maria','Jordan','La Belleza','La Paz',
        'Landazuri','Lebrija','Los Santos','Macaravita','Malaga','Matanza','Mogotes','Molagavita',
        'Ocamonte','Oiba','Onzaga','Palmar','Palmas del Socorro','Paramo','Piedecuesta','Pinchote',
        'Puente Nacional','Puerto Parra','Puerto Wilches','Rionegro','Sabana de Torres',
        'San Andres','San Benito','San Gil','San Joaquin','San Jose de Miranda','San Miguel',
        'San Vicente de Chucuri','Santa Barbara','Santa Helena del Opon','Simacota','Suaita',
        'Sucre','Surata','Tona','Valle de San Jose','Velez','Vetas','Villanueva','Zapatoca'
    ],
    'Sucre': [
        'Buenavista','Caimito','Chalan','Coloso','Corozal','Covenas','El Roble','Galeras',
        'Guaranda','La Union','Los Palmitos','Majagual','Morroa','Ovejas','Sampues',
        'San Antonio de Palmito','San Benito Abad','San Juan de Betulia','San Marcos',
        'San Onofre','San Pedro','Since','Sincelejo','Sucre','Tolu','Tolu Viejo'
    ],
    'Tolima': [
        'Alpujarra','Alvarado','Ambalema','Anzoategui','Armero','Ataco','Cajamarca',
        'Carmen de Apicala','Casabianca','Chaparral','Coello','Coyaima','Cunday','Dolores',
        'El Espinal','Falan','Flandes','Fresno','Guamo','Herveo','Honda','Ibague','Icononzo',
        'Lerida','Libano','Mariquita','Melgar','Murillo','Natagaima','Ortega','Palocabildo',
        'Piedras','Planadas','Prado','Purificacion','Rioblanco','Roncesvalles','Rovira',
        'Saldana','San Antonio','San Luis','Santa Isabel','Suarez','Valle de San Juan',
        'Venadillo','Villahermosa','Villarrica'
    ],
    'Valle del Cauca': [
        'Alcala','Andalucia','Ansermanuevo','Argelia','Bolivar','Buenaventura','Buga',
        'Bugalagrande','Caicedonia','Cali','Calima','Candelaria','Cartago','Dagua',
        'El Aguila','El Cairo','El Cerrito','El Dovio','Florida','Ginebra','Guacari',
        'Jamundi','La Cumbre','La Union','La Victoria','Obando','Palmira','Pradera',
        'Restrepo','Riofrio','Roldanillo','San Pedro','Sevilla','Toro','Trujillo',
        'Tulua','Ulloa','Versalles','Vijes','Yotoco','Yumbo','Zarzal'
    ],
    'Vaupes': ['Caruru','Mitu','Taraira'],
    'Vichada': ['Cumaribo','La Primavera','Puerto Carreno','Santa Rosalia'],
}

In [ ]:
ciudad_a_departamento = {
    normalizar_texto(ciudad): depto
    for depto, ciudades in DEPARTAMENTOS_CIUDADES.items()
    for ciudad in ciudades
}

df_final['CIUDAD_NORM'] = df_final['Ciudad'].apply(normalizar_texto)
df_final['CIUDAD_STD']  = (
    df_final['CIUDAD_NORM']
    .map(MAP_CIUDADES)
    .fillna(df_final['CIUDAD_NORM'])
    .apply(capitalizar_ciudad)
)
df_final['DEPARTAMENTO'] = df_final['CIUDAD_STD'].apply(normalizar_texto).map(ciudad_a_departamento)

In [ ]:
MAP_REGION = {
    'Meta':'ORINOQUIA','Arauca':'ORINOQUIA','Casanare':'ORINOQUIA','Vichada':'ORINOQUIA',
    'Atlantico':'CARIBE','Bolivar':'CARIBE','Cesar':'CARIBE','Cordoba':'CARIBE',
    'La Guajira':'CARIBE','Magdalena':'CARIBE','Sucre':'CARIBE',
    'Valle del Cauca':'PACIFICO','Cauca':'PACIFICO','Narino':'PACIFICO','Choco':'PACIFICO',
    'Caqueta':'AMAZONIA','Putumayo':'AMAZONIA','Amazonas':'AMAZONIA',
    'Guainia':'AMAZONIA','Guaviare':'AMAZONIA','Vaupes':'AMAZONIA',
    'Antioquia':'ANDINA','Boyaca':'ANDINA','Caldas':'ANDINA','Cundinamarca':'ANDINA',
    'Huila':'ANDINA','Norte de Santander':'ANDINA','Quindio':'ANDINA',
    'Risaralda':'ANDINA','Santander':'ANDINA','Tolima':'ANDINA',
    'San Andres':'INSULAR',
}

df_final['REGION'] = df_final['DEPARTAMENTO'].map(MAP_REGION).fillna('OTROS')
print(df_final['REGION'].value_counts().to_string())

## 5. Ingeniería de features

### Bandas actuariales de edad — estándar sector asegurador (FASECOLDA / Swiss Re)

| # | Rango | Etiqueta | Perfil de riesgo |
|---|-------|----------|-----------------|
| 1 | 0 – 17 | **Menor** | Dependiente, siniestralidad baja |
| 2 | 18 – 25 | **Joven** | Inicio vida laboral, riesgo moderado |
| 3 | 26 – 35 | **Adulto Joven** | Formación familia, riesgo bajo-medio |
| 4 | 36 – 45 | **Adulto** | Productividad máxima, riesgo medio |
| 5 | 46 – 55 | **Adulto Mayor** | Inicio enfermedades crónicas, riesgo medio-alto |
| 6 | 56 – 65 | **Pre-Jubilado** | Alta siniestralidad, mayor uso de beneficios |
| 7 | 66 – 75 | **Jubilado** | Riesgo alto, enfermedades degenerativas |
| 8 | 76 – 130 | **Tercera Edad** | Muy alta siniestralidad, dependencia |

> **Implementación:**   
> El parámetro  incluye edad = 0 en la banda **Menor**.  
> La columna  codifica ordinalmente (1 = Menor … 8 = Tercera Edad) para K-Prototypes.


In [ ]:
# Recalcular edad desde FechaNacimiento
df_final['Fechanacimiento'] = pd.to_datetime(
    df_final['Fechanacimiento'].astype(str).str.strip(), errors='coerce', dayfirst=True
)
hoy = pd.Timestamp.today().normalize()
edad_calc = (
    hoy.year - df_final['Fechanacimiento'].dt.year
    - (
        (hoy.month < df_final['Fechanacimiento'].dt.month)
        | ((hoy.month == df_final['Fechanacimiento'].dt.month)
           & (hoy.day < df_final['Fechanacimiento'].dt.day))
    ).astype('Int64')
)
df_final['Edad'] = edad_calc.where((edad_calc >= 0) & (edad_calc <= 130), df_final['Edad']).astype('Int64')

# Valor total del plan
df_final['Valortotalplan'] = (
    pd.to_numeric(df_final['Valormensual'], errors='coerce').fillna(0)
    * pd.to_numeric(df_final['Cuotas'], errors='coerce').fillna(0)
)

# Rango de afiliados por empresa (Nit)
afiliados_x_entidad = (
    df_final.groupby('Nit')['Numeroidentificacion']
    .nunique().reset_index()
    .rename(columns={'Numeroidentificacion': 'Total_afiliados'})
)
df_final = df_final.merge(afiliados_x_entidad, on='Nit', how='left')

RANGOS_AFIL = [(100,'0-100'),(300,'100-300'),(500,'300-500'),(1000,'500-1000'),
               (2000,'1000-2000'),(5000,'2000-5000'),(10000,'5000-10000')]

def asignar_rango_afiliados(n):
    for limite, label in RANGOS_AFIL:
        if n <= limite:
            return label
    return 'Mas 10.000'

df_final['Rango_afiliados'] = df_final['Total_afiliados'].apply(asignar_rango_afiliados)

# Mascotas
df_final['Cantidad_mascotas'] = (
    df_final['Edadesmascotas'].astype(str)
    .str.split('-').apply(lambda x: len([i for i in x if i.strip() != '']) if isinstance(x, list) else 0)
)
listas_edades = df_final['Edadesmascotas'].apply(
    lambda v: [int(x.strip()) for x in str(v).split('-') if x.strip().isdigit()]
    if not pd.isna(v) and str(v).strip() != '' else []
)
df_final['PromEdadMascotas'] = listas_edades.apply(
    lambda x: round(sum(x)/len(x), 1) if x else np.nan
)

# Normalizar Sexo
def normalizar_sexo(x):
    if pd.isna(x): return pd.NA
    x = ''.join(c for c in unicodedata.normalize('NFD', str(x).strip().upper())
                if unicodedata.category(c) != 'Mn')
    if x in {'M','MASCULINO','HOMBRE'}: return 'M'
    if x in {'F','FEMENINO','MUJER'}:   return 'F'
    return pd.NA

df_final['Sexo']    = df_final['Sexo'].apply(normalizar_sexo)
df_final['Estrato'] = df_final['Estrato'].replace(9, 0)

# Clasificar TiposSeguros
def clasificar_seguro(val):
    if pd.isna(val): return np.nan
    v = val.upper()
    if 'A. P.' in v or 'ACCIDENTES PERSONALES' in v: return 'AP'
    if 'SINERGIA' in v: return 'PFI'
    return 'SOLICANASTA'

df_final['Tiposseguros_ajuste'] = df_final['Tiposseguros'].apply(clasificar_seguro)

# Rango de edad — bandas actuariales sector asegurador (FASECOLDA)
BINS_EDAD   = [0, 17, 25, 35, 45, 55, 65, 75, 130]
LABELS_EDAD = ['Menor', 'Joven', 'Adulto Joven', 'Adulto',
               'Adulto Mayor', 'Pre-Jubilado', 'Jubilado', 'Tercera Edad']
df_final['Rango_edad'] = pd.cut(
    df_final['Edad'], bins=BINS_EDAD, labels=LABELS_EDAD,
    right=True, include_lowest=True
).astype(str)

print('Features creadas correctamente.')

## 6. Selección y ordenación de columnas

In [ ]:
# Seleccion y orden de columnas finales
COLUMNAS_FINALES = [
    'Contrato','Estado','Nroentidad','Nit','Entidad','Nombrecompleto',
    'Numeroidentificacion','Edad','Fechanacimiento','Sexo','Estadocivil','Estrato',
    'Direccion','Telefono','Correo','Fechaingreso','Tienepadres','Tieneesposa',
    'Tienehijos','Tieneperro','Tienegato','Edadesmascotas','Cantidad_mascotas',
    'PromEdadMascotas','Razasmascotas','Producto','Canal','Tiposseguros',
    'Tiposseguros_ajuste','Ciudad','Codigoproducto','Nombreproducto',
    'Valormensual','Act.valor','Periodicidad','Cuotas','Valortotalplan',
    'Contratante','Antigedad','Rango_antigedad','Total_afiliados','Rango_afiliados',
    'Rango_edad','Rango_tarifa','Rango_produccion','Rango_mortalidad',
    'Rango_siniestralidad','Rentabilidad','Top_resultados','Tipo_entidad_1','Tipo_entidad_2',
    'CIUDAD_NORM','CIUDAD_STD','DEPARTAMENTO','REGION'
]
COLUMNAS_FINALES = [c for c in COLUMNAS_FINALES if c in df_final.columns]
df_final = df_final[COLUMNAS_FINALES].copy()
df_final = df_final.rename(columns={
    'Tipo_entidad_1': 'SECTOR_EMPLEADOR',
    'Tipo_entidad_2': 'ACTIVIDAD_ECONOMICA',
})
print(f'Dataset final: {df_final.shape}')

## 7. Análisis exploratorio (EDA)

In [ ]:
# Perfil estructural del dataset
perfil = pd.DataFrame({
    'Tipo_Dato'      : df_final.dtypes,
    'No_Nulos'       : df_final.count(),
    'Nulos'          : df_final.isnull().sum(),
    'Pct_Nulos'      : (df_final.isnull().mean() * 100).round(2),
    'Valores_Unicos' : df_final.nunique(),
})
perfil.sort_values('Pct_Nulos', ascending=False)

In [ ]:
# Estadisticas de variables numericas clave
df_final[['Edad','Valortotalplan','Total_afiliados','Cantidad_mascotas','PromEdadMascotas']].describe().T

In [ ]:
# Distribucion de variables categoricas
vars_cat = {
    'Sexo'        : df_final['Sexo'],
    'Estadocivil' : df_final['Estadocivil'],
    'Estrato'     : df_final['Estrato'],
    'REGION'      : df_final['REGION'],
    'Producto'    : df_final['Producto'],
    'Tienepadres' : df_final['Tienepadres'],
    'Tieneesposa' : df_final['Tieneesposa'],
    'Tienehijos'  : df_final['Tienehijos'],
    'Tieneperro'  : df_final['Tieneperro'],
    'Tienegato'   : df_final['Tienegato'],
}

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for idx, (ax, (nombre, serie)) in enumerate(zip(axes.flatten(), vars_cat.items())):
    conteo = serie.value_counts(normalize=True).head(8) * 100
    n = len(conteo)
    colores = [PALETTE[i % len(PALETTE)] for i in range(n)]
    conteo.plot(kind='bar', ax=ax, color=colores, edgecolor='white')
    ax.set_title(nombre, fontsize=11, fontweight='bold')
    ax.set_ylabel('%')
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Distribucion de variables categoricas', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Distribucion de Valortotalplan
serie_vtp = df_final.loc[df_final['Valortotalplan'] > 0, 'Valortotalplan']
q1, q2, q3 = serie_vtp.quantile([0.25, 0.50, 0.75])
iqr = q3 - q1
print(f'Min:{int(serie_vtp.min()):,} | Q1:{int(q1):,} | Med:{int(q2):,} | Q3:{int(q3):,} | Max:{int(serie_vtp.max()):,} | IQR:{int(iqr):,}')

bp_kw = dict(vert=True, patch_artist=True, boxprops=dict(facecolor=PALETTE[0], alpha=0.6))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].boxplot(serie_vtp, **bp_kw)
axes[0].set_title('Escala original')

axes[1].boxplot(serie_vtp, showfliers=False, **bp_kw)
axes[1].set_title('Sin outliers')

axes[2].boxplot(serie_vtp, **bp_kw)
axes[2].set_yscale('log')
axes[2].set_title('Escala log')

for ax in axes:
    ax.set_ylabel('Valor Total Plan ($)')

plt.suptitle('Distribucion Valor Total del Plan (sin ceros)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
scaler = StandardScaler()

df_modelo = df_final.copy()
df_modelo.columns = df_modelo.columns.str.strip()
df_modelo['ValorTotal_scaled'] = scaler.fit_transform(df_modelo[['Valortotalplan']])

# Codificacion numerica auxiliar para visualizacion (K-Prototypes usa categoricas directas)
_reg_map = {'AMAZONIA':0,'ANDINA':1,'CARIBE':2,'ORINOQUIA':3,'OTROS':4,'PACIFICO':5,'INSULAR':6}
_ec_map  = {'CAS':0,'DIV':1,'OTR':2,'SEP':3,'SOL':4,'UNI':5,'VIU':6}

df_modelo['TienePadres_V'] = (df_modelo['Tienepadres'] == 'Y').astype(int)
df_modelo['TieneEsposa_V'] = (df_modelo['Tieneesposa'] == 'Y').astype(int)
df_modelo['TieneHijos_V']  = (df_modelo['Tienehijos']  == 'Y').astype(int)
df_modelo['EstadoCivil_V'] = df_modelo['Estadocivil'].map(_ec_map).fillna(2).astype(int)
df_modelo['Region_V']      = df_modelo['REGION'].map(_reg_map).fillna(4).astype(int)

# ── Flags de cobertura desde PAP_clientes (columnas binarias, no en df_final) ─
_COBERTURAS = ['Salud', 'Bicicleta', 'Repatriacion', 'Expatriacion']
_cob_ref = (
    PAP_clientes
    .assign(**{col: PAP_clientes[col].notna() & (PAP_clientes[col].astype(str).str.strip() != '')
               for col in _COBERTURAS})
    .groupby('Contrato')[_COBERTURAS]
    .any()
    .astype(int)
    .reset_index()
)
_cob_ref['Contrato'] = _cob_ref['Contrato'].astype(str)
df_modelo['Contrato'] = df_modelo['Contrato'].astype(str)
df_modelo = df_modelo.merge(_cob_ref, on='Contrato', how='left')
for col in _COBERTURAS:
    df_modelo[col] = df_modelo[col].fillna(0).astype(int)

# Subset para K-Prototypes
COLS_KPROTOTYPES = [
    'Sexo','Edad','Estadocivil','Tienepadres','Tieneesposa',
    'Tienehijos','Tieneperro','Tienegato','Canal','REGION','ValorTotal_scaled'
]
df_kprototypes = df_modelo[COLS_KPROTOTYPES].copy()

print(f'df_modelo: {df_modelo.shape}')
print(f'df_kprototypes: {df_kprototypes.shape}')
print(df_kprototypes.dtypes)

In [ ]:
# Visualizacion: Edad vs ValorTotal por variables de composicion familiar
fig, axes_arr = plt.subplots(1, 3, figsize=(18, 5))

sample = df_modelo.sample(min(10000, len(df_modelo)), random_state=42)

for ax, (hue_col, titulo) in zip(axes_arr, [
    ('TienePadres_V', 'Tiene Padres'),
    ('TieneEsposa_V', 'Tiene Esposa'),
    ('TieneHijos_V',  'Tiene Hijos'),
]):
    sns.scatterplot(
        data=sample, x='Edad', y='ValorTotal_scaled',
        hue=hue_col, palette=[PALETTE[0], PALETTE[2]],
        alpha=0.3, s=10, ax=ax
    )
    ax.set_title(titulo, fontweight='bold')
    ax.legend(title=hue_col.replace('_V', ''), labels=['No', 'Si'])

plt.suptitle('Edad vs Valor Total por composicion familiar', fontsize=13)
plt.tight_layout()
plt.show()

sns.pairplot(
    data=sample[['Edad', 'ValorTotal_scaled', 'Region_V', 'EstadoCivil_V']],
    hue='EstadoCivil_V', corner=True,
    plot_kws={'alpha': 0.3, 's': 10}
)
plt.suptitle('Pairplot por Estado Civil', y=1.01)
plt.show()

## 8. Exportación

In [ ]:
df_modelo.to_csv("data_limpia.csv", index=False)
print(f"Exportado: data_limpia.csv — {df_modelo.shape[0]:,} registros, {df_modelo.shape[1]} columnas")
print(f"Act.valor total: {df_modelo['Act.valor'].sum():,.0f}  |  registros: {len(df_modelo):,}")

## 9. Estadísticas empresariales y de familias

### 9.1 Información empresas (NITs)

In [ ]:
# KPI: Empresas únicas (NIT)
nits_unicos = df_modelo['Nit'].nunique()

fig, ax = plt.subplots(figsize=(5, 3.2))
fig.patch.set_facecolor('white')
ax.set_facecolor('#eef7f1')

ax.text(0.5, 0.58, f'{nits_unicos:,}',
        fontsize=54, ha='center', va='center',
        fontweight='bold', color=C_VERDE, transform=ax.transAxes)
ax.text(0.5, 0.22, 'Empresas únicas registradas',
        fontsize=12, ha='center', va='center',
        color=C_V_PAL, transform=ax.transAxes)

ax.set_xticks([]); ax.set_yticks([])
for sp in ax.spines.values():
    sp.set_edgecolor(C_VERDE); sp.set_linewidth(2)
ax.set_title('Indicador de Empresas Únicas (NIT)', pad=14)
plt.tight_layout()
plt.show()

In [ ]:
ORDEN_RANGO = ['0-100','100-300','300-500','500-1000','1000-2000','2000-5000','5000-10000','Mas 10.000']

nits_por_tamano = (
    df_modelo.groupby('Rango_afiliados')['Nit']
    .nunique()
    .reindex(ORDEN_RANGO)
    .dropna()
)

n = len(nits_por_tamano)
colores = [PALETTE[i % len(PALETTE)] for i in range(n)]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(range(n), nits_por_tamano.values,
              color=colores, edgecolor='white', linewidth=0.8)

for bar, v in zip(bars, nits_por_tamano.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f'{int(v):,}', ha='center', va='bottom',
            fontsize=10, color=C_VERDE, fontweight='bold')

ax.set_xticks(range(n))
ax.set_xticklabels(nits_por_tamano.index, rotation=30, ha='right')
ax.set_title('Empresas unicas (NIT) por rango de afiliados')
ax.set_ylabel('Empresas (NIT unicos)')
plt.tight_layout()
plt.show()

In [ ]:
nits_por_canal = df_modelo.groupby('Canal')['Nit'].nunique().sort_values(ascending=False)

n = len(nits_por_canal)
colores = [PALETTE[i % len(PALETTE)] for i in range(n)]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(range(n), nits_por_canal.values,
              color=colores, edgecolor='white', linewidth=0.8)

for bar, v in zip(bars, nits_por_canal.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f'{int(v):,}', ha='center', va='bottom',
            fontsize=10, color=C_VERDE, fontweight='bold')

ax.set_xticks(range(n))
ax.set_xticklabels(nits_por_canal.index, rotation=35, ha='right')
ax.set_title('Empresas unicas (NIT) por Canal comercial')
ax.set_ylabel('Empresas (NIT unicos)')
plt.tight_layout()
plt.show()

In [ ]:
depto_counts = df_modelo.groupby('DEPARTAMENTO')['Contrato'].nunique().sort_values(ascending=False).head(10)

n = len(depto_counts)
colores = [PALETTE[i % len(PALETTE)] for i in range(n)]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(range(n), depto_counts.values,
              color=colores, edgecolor='white', linewidth=0.8)

for bar, v in zip(bars, depto_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f'{int(v):,}', ha='center', va='bottom',
            fontsize=10, color=C_VERDE, fontweight='bold')

ax.set_xticks(range(n))
ax.set_xticklabels(depto_counts.index, rotation=35, ha='right')
ax.set_title('Top 10 Departamentos por Contratos unicos')
ax.set_ylabel('Contratos unicos')
plt.tight_layout()
plt.show()

In [ ]:
ciudad_counts = df_modelo.groupby('CIUDAD_STD')['Contrato'].nunique().sort_values(ascending=False).head(10)

n = len(ciudad_counts)
colores = [PALETTE[i % len(PALETTE)] for i in range(n)]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(range(n), ciudad_counts.values,
              color=colores, edgecolor='white', linewidth=0.8)

for bar, v in zip(bars, ciudad_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f'{int(v):,}', ha='center', va='bottom',
            fontsize=10, color=C_VERDE, fontweight='bold')

ax.set_xticks(range(n))
ax.set_xticklabels(ciudad_counts.index, rotation=35, ha='right')
ax.set_title('Top 10 Ciudades por Contratos unicos')
ax.set_ylabel('Contratos unicos')
plt.tight_layout()
plt.show()

In [ ]:
depto = (
    df_modelo.groupby('DEPARTAMENTO')['Act.valor']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

n = len(depto)
colores = [PALETTE[i % len(PALETTE)] for i in range(n)]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(range(n), depto.values,
              color=colores, edgecolor='white', linewidth=0.8)

for bar, v in zip(bars, depto.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 80,
            f'{v:,.0f}', ha='center', va='bottom',
            fontsize=9, color=C_VERDE, fontweight='bold')

ax.set_xticks(range(n))
ax.set_xticklabels(depto.index, rotation=35, ha='right')
ax.set_title('Top 10 Departamentos por familias afiliadas')
ax.set_ylabel('Familias (Act.valor)')
plt.tight_layout()
plt.show()

In [ ]:
canal = df_modelo.groupby('Canal')['Act.valor'].sum().sort_values(ascending=False)

n = len(canal)
colores = [PALETTE[i % len(PALETTE)] for i in range(n)]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(range(n), canal.values,
              color=colores, edgecolor='white', linewidth=0.8)

for bar, v in zip(bars, canal.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 80,
            f'{v:,.0f}', ha='center', va='bottom',
            fontsize=9, color=C_VERDE, fontweight='bold')

ax.set_xticks(range(n))
ax.set_xticklabels(canal.index, rotation=35, ha='right')
ax.set_title('Familias afiliadas por Canal comercial')
ax.set_ylabel('Familias (Act.valor)')
plt.tight_layout()
plt.show()

In [ ]:
ciudad = df_modelo.groupby('CIUDAD_STD')['Act.valor'].sum().sort_values(ascending=False).head(10)

n = len(ciudad)
colores = [PALETTE[i % len(PALETTE)] for i in range(n)]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(range(n), ciudad.values,
              color=colores, edgecolor='white', linewidth=0.8)

for bar, v in zip(bars, ciudad.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 40,
            f'{v:,.0f}', ha='center', va='bottom',
            fontsize=9, color=C_VERDE, fontweight='bold')

ax.set_xticks(range(n))
ax.set_xticklabels(ciudad.index, rotation=35, ha='right')
ax.set_title('Top 10 Ciudades por familias afiliadas')
ax.set_ylabel('Familias (Act.valor)')
plt.tight_layout()
plt.show()

In [ ]:
sexo_dist = df_modelo['Sexo'].value_counts(normalize=True).mul(100).round(2)
print(sexo_dist.to_string())

n = len(sexo_dist)
colores = [PALETTE[i % len(PALETTE)] for i in range(n)]
fig, ax = plt.subplots(figsize=(6, 5))
sexo_dist.plot(
    kind='pie', ax=ax, colors=colores,
    autopct='%1.1f%%', startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2.5),
    textprops={'fontsize': 12, 'color': 'white', 'fontweight': 'bold'},
    pctdistance=0.68,
)
ax.set_ylabel('')
ax.set_title('Distribucion por sexo', pad=14)
plt.tight_layout()
plt.show()

In [ ]:
ORDEN_EDAD = ['Menor', 'Joven', 'Adulto Joven', 'Adulto',
              'Adulto Mayor', 'Pre-Jubilado', 'Jubilado', 'Tercera Edad']
orden_presente = [c for c in ORDEN_EDAD if c in df_modelo['Rango_edad'].unique()]

cruce = (pd.crosstab(df_modelo['Rango_edad'], df_modelo['Sexo'])
           .reindex(orden_presente, fill_value=0))
totales = cruce.sum(axis=1)

n_cols = cruce.shape[1]
colores = [PALETTE[i % len(PALETTE)] for i in range(n_cols)]

fig, ax = plt.subplots(figsize=(12, 6))
cruce.plot(kind='bar', stacked=True, ax=ax,
           color=colores, edgecolor='white', linewidth=0.8)

# Etiquetas dentro de cada segmento (solo si >= 5% del total de la barra)
for i, (idx, row) in enumerate(cruce.iterrows()):
    acumulado = 0
    total_barra = row.sum()
    for j, val in enumerate(row):
        if val > 0:
            pct = val / total_barra * 100
            if pct >= 5:
                ax.text(i, acumulado + val / 2,
                        f'{int(val):,}({pct:.0f}%)',
                        ha='center', va='center',
                        fontsize=8, color='white', fontweight='bold')
            acumulado += val
    # Total encima de la barra
    ax.text(i, total_barra + totales.max() * 0.012,
            f'{int(total_barra):,}',
            ha='center', va='bottom', fontsize=9, color=C_VERDE, fontweight='bold')

ax.set_title('Distribucion por sexo y rango de edad', pad=12)
ax.set_xlabel('Rango de edad')
ax.set_ylabel('Cantidad de personas')
ax.set_xticklabels(orden_presente, rotation=35, ha='right')
ax.grid(axis='y', alpha=0.2)
ax.legend(title='Sexo', loc='upper right')
plt.tight_layout()
plt.show()


In [ ]:
# Orden cronologico filtrado — bandas actuariales (FASECOLDA)
ORDEN_EDAD = [c for c in ORDEN_EDAD if c in df_modelo['Rango_edad'].unique()]
edad_dist  = df_modelo['Rango_edad'].value_counts().reindex(ORDEN_EDAD).dropna()

# Codificacion numerica ordinal (1=Menor ... 8=Tercera Edad)
mapa_edad_n = {cat: i + 1 for i, cat in enumerate(ORDEN_EDAD)}
df_modelo['Rango_edad_N'] = df_modelo['Rango_edad'].map(mapa_edad_n)
print(edad_dist.to_string())

n = len(edad_dist)
colores = [PALETTE[i % len(PALETTE)] for i in range(n)]
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(n), edad_dist.values, color=colores, edgecolor='white', linewidth=0.8)
offset = edad_dist.max() * 0.012
for bar, v in zip(bars, edad_dist.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + offset,
            f'{v:,}', ha='center', va='bottom', fontsize=10, color=C_VERDE, fontweight='bold')
ax.set_xticks(range(n))
ax.set_xticklabels(edad_dist.index, rotation=30, ha='right')
ax.set_title('Distribucion de afiliados por rango de edad')
ax.set_ylabel('Cantidad de personas')
plt.tight_layout()
plt.show()

In [ ]:
# ── Normalización de flags y features de producto ──────────────────────────
# Tieneperro / Tienegato: NaN → 'N'
df_modelo['Tieneperro'] = (
    df_modelo['Tieneperro'].astype(str).str.strip().str.upper()
    .replace({'NAN': 'N', 'NONE': 'N', '': 'N'})
)
df_modelo['Tienegato'] = (
    df_modelo['Tienegato'].astype(str).str.strip().str.upper()
    .replace({'NAN': 'N', 'NONE': 'N', '': 'N'})
)
# tiene_mascotas
df_modelo['tiene_mascotas'] = df_modelo['Tieneperro'].eq('Y') | df_modelo['Tienegato'].eq('Y')

# tiene_poliza: isin con valores válidos reales — robusto frente a NaN / 'nan' strings
TIPOS_POLIZA_VALIDOS = ['AP', 'PFI', 'SOLICANASTA']
df_modelo['tiene_poliza'] = df_modelo['Tiposseguros_ajuste'].isin(TIPOS_POLIZA_VALIDOS)

print(f'Total registros : {len(df_modelo):,}')
print(f'tiene_mascotas  : {df_modelo["tiene_mascotas"].sum():,}  ({df_modelo["tiene_mascotas"].mean()*100:.1f}%)')
print(f'tiene_poliza    : {df_modelo["tiene_poliza"].sum():,}  ({df_modelo["tiene_poliza"].mean()*100:.1f}%)')
print(f'Canales distintos: {df_modelo["Canal"].nunique()}')


### 9.5. Segmentación adicional

In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# df_tipo — nivel contrato, clasificación por 6 productos
# (Póliza | Salud | Bicicleta | Repatriacion | Expatriacion | Mascotas)
# Fuente: df_modelo (incluye columnas de cobertura merged desde PAP_clientes)
# ══════════════════════════════════════════════════════════════════════════════

_PRODS_6 = ['Salud', 'Bicicleta', 'Repatriacion', 'Expatriacion']
_SEIS    = ['Poliza', 'Salud', 'Bicicleta', 'Repatriacion', 'Expatriacion', 'Mascotas']

# ── Una fila por contrato ─────────────────────────────────────────────────────
df_tipo = (
    df_modelo
    .groupby(['Contrato', 'Canal'], dropna=False)
    .agg({**{p: 'max' for p in _PRODS_6},
          'tiene_poliza':   'max',
          'tiene_mascotas': 'max'})
    .reset_index()
)
df_tipo['Poliza']   = df_tipo['tiene_poliza'].astype(int)
df_tipo['Mascotas'] = df_tipo['tiene_mascotas'].astype(int)
for col in _PRODS_6:
    df_tipo[col] = df_tipo[col].astype(int)

df_tipo['n_productos'] = df_tipo[_SEIS].sum(axis=1)

def _etiqueta6(row):
    items = [s for s in _SEIS if row[s] == 1]
    return ' + '.join(items) if items else 'Sin productos'

df_tipo['Tipo_asistencia'] = df_tipo.apply(_etiqueta6, axis=1)

print(f'Contratos únicos: {len(df_tipo):,}')
print(df_tipo['Tipo_asistencia'].value_counts().head(20).to_string())

# ── Totales por dimensión ─────────────────────────────────────────────────────
total_tipo = len(df_tipo)
print(f"\n{'─'*60}")
print(f"  TOTAL CONTRATOS POR DIMENSIÓN ({total_tipo:,} contratos únicos)")
print(f"{'─'*60}")
for dim in _SEIS:
    tot = int(df_tipo[dim].sum())
    print(f"  {dim:<15}  {tot:>10,}   ({tot/total_tipo*100:.1f}%)")

# ── Gráfico: canal vs tipo de asistencia (top combinaciones) ─────────────────
# Agrupamos las combinaciones con < 0.5% del total en "Otras combinaciones"
_umbral  = total_tipo * 0.005
_counts  = df_tipo['Tipo_asistencia'].value_counts()
_top     = _counts[_counts >= _umbral].index.tolist()
_otros   = _counts[_counts < _umbral].index.tolist()

df_tipo_plot = df_tipo.copy()
if _otros:
    df_tipo_plot.loc[df_tipo_plot['Tipo_asistencia'].isin(_otros), 'Tipo_asistencia'] = 'Otras combinaciones'

cols_pres = (
    df_tipo_plot['Tipo_asistencia']
    .value_counts()
    .sort_values(ascending=False)
    .index.tolist()
)
# Mover "Otras combinaciones" y "Sin productos" al final
for _last in ['Sin productos', 'Otras combinaciones']:
    if _last in cols_pres:
        cols_pres.remove(_last)
        cols_pres.append(_last)

cruce = (
    df_tipo_plot
    .groupby(['Canal', 'Tipo_asistencia'])['Contrato']
    .nunique()
    .unstack(fill_value=0)
    .reindex(columns=cols_pres, fill_value=0)
)
cruce = cruce.loc[cruce.sum(axis=1).sort_values(ascending=False).index]
totales = cruce.sum(axis=1)
print(f'\nTotal contratos en cruce: {int(totales.sum()):,}')

# Colores: ciclar PALETTE por columna
_ncols     = len(cruce.columns)
_colors    = [PALETTE[i % len(PALETTE)] for i in range(_ncols)]
TEXT_COLORS = ['white', 'white', '#333333', '#333333']

fig, ax = plt.subplots(figsize=(14, 7))
cruce.plot(kind='bar', stacked=True, ax=ax,
           color=_colors,
           edgecolor='white', linewidth=0.6, width=0.75)

for col_idx, container in enumerate(ax.containers):
    txt_color = TEXT_COLORS[col_idx % len(TEXT_COLORS)]
    for bar_idx, bar in enumerate(container):
        h = bar.get_height()
        total_b = totales.iloc[bar_idx]
        if h > 0 and h / total_b >= 0.04:
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_y() + h / 2,
                    f'{int(h):,}',
                    ha='center', va='center',
                    fontsize=8.5, fontweight='bold', color=txt_color)

for i, total in enumerate(totales):
    ax.text(i, total + totales.max() * 0.015,
            f'{int(total):,}',
            ha='center', va='bottom',
            fontsize=9, fontweight='bold', color=C_VERDE)

ax.set_title('Canal comercial vs Tipo de asistencia — 6 productos\n(combinaciones exactas, contratos únicos)',
             fontweight='bold')
ax.set_ylabel('Contratos')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=35)
plt.setp(ax.get_xticklabels(), ha='right')
ax.legend(title='Combinación', bbox_to_anchor=(1.01, 1),
          loc='upper left', frameon=True, framealpha=0.9, fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PENETRACIÓN — Segmentación exacta por combinación de productos
# Fuente: df_modelo
# (Póliza | Salud | Bicicleta | Repatriacion | Expatriacion | Mascotas)
# ══════════════════════════════════════════════════════════════════════════════

PRODUCTOS = ['Salud', 'Bicicleta', 'Repatriacion', 'Expatriacion']
SEIS      = ['Poliza', 'Salud', 'Bicicleta', 'Repatriacion', 'Expatriacion', 'Mascotas']

# ── Una fila por contrato desde df_modelo ────────────────────────────────────
df_penet = (
    df_modelo
    .groupby('Contrato')[PRODUCTOS + ['tiene_poliza', 'tiene_mascotas']]
    .agg({**{p: 'max' for p in PRODUCTOS},
          'tiene_poliza': 'any',
          'tiene_mascotas': 'any'})
    .reset_index()
)
df_penet['Poliza']   = df_penet['tiene_poliza'].astype(int)
df_penet['Mascotas'] = df_penet['tiene_mascotas'].astype(int)
for col in PRODUCTOS:
    df_penet[col] = df_penet[col].astype(int)

df_penet['n_productos'] = df_penet[SEIS].sum(axis=1)

def _etiqueta(row):
    items = [s for s in SEIS if row[s] == 1]
    return ' + '.join(items) if items else 'Sin productos'

df_penet['Etiqueta'] = df_penet.apply(_etiqueta, axis=1)

total_penet = len(df_penet)

combos = (
    df_penet.groupby(['n_productos', 'Etiqueta'])
    .size()
    .reset_index(name='Contratos')
    .sort_values(['n_productos', 'Contratos'], ascending=[False, False])
    .reset_index(drop=True)
)
combos['% del total'] = (combos['Contratos'] / total_penet * 100).round(2)

# ── Totales independientes por dimensión ──────────────────────────────────────
totales_dim = {dim: int(df_penet[dim].sum()) for dim in SEIS}

# ── Gráfico 2 paneles ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(20, max(6, len(combos[combos['Contratos']>0]) * 0.45)))

# Panel izquierdo — combinaciones exactas (mayor a menor)
top_combos = (
    combos[combos['Contratos'] > 0]
    .sort_values('Contratos', ascending=True)
    .reset_index(drop=True)
)
color_map  = {6: PALETTE[0], 5: PALETTE[1], 4: PALETTE[2], 3: PALETTE[3],
              2: PALETTE[0], 1: PALETTE[1], 0: PALETTE[2]}
colores_bar = [color_map.get(n, PALETTE[0]) for n in top_combos['n_productos']]

ax1 = axes[0]
bars1 = ax1.barh(top_combos['Etiqueta'], top_combos['Contratos'],
                 color=colores_bar, edgecolor='white', linewidth=0.6)
for bar, v, pct in zip(bars1, top_combos['Contratos'], top_combos['% del total']):
    ax1.text(bar.get_width() + top_combos['Contratos'].max() * 0.01,
             bar.get_y() + bar.get_height() / 2,
             f'{v:,}  ({pct:.1f}%)',
             va='center', fontsize=8, fontweight='bold', color=C_VERDE)
ax1.set_xlabel('Contratos únicos')
ax1.set_title('Combinaciones exactas\n(de mayor a menor)', fontweight='bold')
ax1.set_xlim(0, top_combos['Contratos'].max() * 1.22)

# Panel derecho — total de contratos por dimensión individual
ax2 = axes[1]
dims  = list(totales_dim.keys())
vals  = list(totales_dim.values())
cols2 = [PALETTE[i % len(PALETTE)] for i in range(len(dims))]
bars2 = ax2.bar(dims, vals, color=cols2, edgecolor='white', linewidth=0.8)
offset2 = max(vals) * 0.015
for bar, v in zip(bars2, vals):
    pct = v / total_penet * 100
    ax2.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + offset2,
             f'{v:,}\n({pct:.1f}%)',
             ha='center', va='bottom', fontsize=9, fontweight='bold', color=C_VERDE)
ax2.set_title('Total contratos por dimensión\n(un contrato puede tener varias)', fontweight='bold')
ax2.set_ylabel('Contratos')
ax2.tick_params(axis='x', rotation=15)

plt.suptitle(f'Participación de mercado  |  {total_penet:,} contratos únicos',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# ── Tabla combinaciones ───────────────────────────────────────────────────────
print(f"{'─'*72}")
print(f"  {'N':>5}   {'Contratos':>10}   {'% total':>7}   Combinación")
print(f"{'─'*72}")
for n in sorted(combos['n_productos'].unique(), reverse=True):
    bloque = combos[combos['n_productos'] == n]
    sufijo  = 'CUMPLE TODOS ✓' if n == 6 else f'{n} de 6'
    subtotal = bloque['Contratos'].sum()
    print(f"\n  ── {sufijo}  (subtotal: {subtotal:,}  |  {subtotal/total_penet*100:.1f}%) ──")
    for _, row in bloque.iterrows():
        print(f"     {int(row['n_productos']):>2} de 6   {row['Contratos']:>10,}   {row['% del total']:>6.1f}%   {row['Etiqueta']}")

print(f"\n{'─'*72}")
print(f"  TOTAL          {total_penet:>10,}   100.0%")

# ── Totales por dimensión (para comparar con gráfico simple) ─────────────────
print(f"\n{'─'*72}")
print(f"  TOTAL CONTRATOS POR DIMENSIÓN  (un contrato puede tener varias)")
print(f"  Nota: sumar estas cifras NO da el total — hay contratos con más de una")
print(f"{'─'*72}")
for dim, tot in totales_dim.items():
    print(f"  {dim:<15}  {tot:>10,}   ({tot/total_penet*100:.1f}%)")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# RESUMEN DE MASCOTAS — Contratos y suma de cantidad
# ══════════════════════════════════════════════════════════════════════════════

mascotas_por_contrato = (
    df_modelo.groupby('Contrato')['Cantidad_mascotas']
    .max()
    .reset_index()
)

total_contratos_masc = (mascotas_por_contrato['Cantidad_mascotas'] > 0).sum()
total_sum_mascotas   = mascotas_por_contrato['Cantidad_mascotas'].sum()

print(f"Contratos con mascotas : {total_contratos_masc:,}")
print(f"Suma total mascotas    : {int(total_sum_mascotas):,}")

# ── Gráfico ───────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))

etiquetas = ['Contratos\ncon mascotas', 'Suma total\nmascotas']
valores   = [int(total_contratos_masc), int(total_sum_mascotas)]
colores   = [PALETTE[0], PALETTE[2]]

bars = ax.bar(etiquetas, valores, color=colores, edgecolor='white', linewidth=0.8, width=0.5)
offset = max(valores) * 0.015
for bar, v in zip(bars, valores):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + offset,
            f'{v:,}', ha='center', va='bottom',
            fontsize=12, fontweight='bold', color=C_VERDE)

ax.set_title('Mascotas — Contratos y cantidad total', fontweight='bold')
ax.set_ylabel('Cantidad')
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MASCOTAS — Análisis detallado
# ══════════════════════════════════════════════════════════════════════════════

_col_edades = next((c for c in df_modelo.columns if c.lower() == 'edadesmascotas'), None)
_col_razas  = next((c for c in df_modelo.columns if c.lower() == 'razasmascotas'), None)

masc_det = (
    df_modelo
    .groupby('Contrato')
    .agg(
        Tieneperro    = ('Tieneperro',       lambda x: 'Y' if 'Y' in x.values else 'N'),
        Tienegato     = ('Tienegato',         lambda x: 'Y' if 'Y' in x.values else 'N'),
        Cantidad_masc = ('Cantidad_mascotas', 'max'),
        Edades_raw    = (_col_edades,         'first'),
        Razas         = (_col_razas,          'first'),
    )
    .reset_index()
)
masc_det = masc_det[masc_det['Cantidad_masc'] > 0].copy()

def _tipo_masc(row):
    if row['Tieneperro'] == 'Y' and row['Tienegato'] == 'Y': return 'Perro + Gato'
    if row['Tieneperro'] == 'Y': return 'Solo Perro'
    if row['Tienegato'] == 'Y':  return 'Solo Gato'
    return 'Sin clasificar'
masc_det['Tipo_mascota'] = masc_det.apply(_tipo_masc, axis=1)

# ── Edades individuales ───────────────────────────────────────────────────────
# Umbral biológico: mascotas no superan los 30 años.
# Las 39 edades > 30 son errores de captura (edad del titular ingresada por error).
EDAD_MAX_MASCOTA = 30

def _es_num(s):
    try: float(s.strip()); return True
    except: return False

def _parsear_edades(val):
    if pd.isna(val): return []
    s = str(val).strip().replace('--', '-').strip('-')
    if not s or s.lower() in ('nan', 'none', ''): return []
    edades = []
    for parte in s.split('-'):
        try:
            e = float(parte.strip())
            if 0 <= e <= EDAD_MAX_MASCOTA:
                edades.append(e)
        except ValueError:
            pass
    return edades

todas_edades = (
    masc_det['Edades_raw']
    .apply(_parsear_edades)
    .explode()
    .dropna()
    .astype(float)
)

total_contratos_masc = len(masc_det)
total_edades         = len(todas_edades)

n_edad_invalida = masc_det['Edades_raw'].apply(
    lambda v: sum(
        1 for p in str(v).replace('--', '-').strip('-').split('-')
        if p.strip() and _es_num(p) and float(p.strip()) > EDAD_MAX_MASCOTA
    ) if not pd.isna(v) else 0
)

# ── Suma de mascotas por tipo (no contratos) ──────────────────────────────────
tipo_sum = (
    masc_det.groupby('Tipo_mascota')['Cantidad_masc']
    .sum()
    .sort_values(ascending=False)
)
total_sum_tipo = tipo_sum.sum()

print(f"Contratos con mascotas : {total_contratos_masc:,}")
print(f"Suma total mascotas    : {int(total_sum_tipo):,}")
print(f"  - Edad válida (≤{EDAD_MAX_MASCOTA} años) : {total_edades:,}")
print(f"  - Edad inválida (>{EDAD_MAX_MASCOTA} años): {int(n_edad_invalida.sum()):,}  ← probable error de captura")
print(f"\nSuma de mascotas por tipo:")
for t, v in tipo_sum.items():
    print(f"  {t:<20}  {int(v):>8,}  ({v/total_sum_tipo*100:.1f}%)")

# ── Gráfico 4 paneles ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Panel 1: Suma de mascotas por tipo
ax1 = axes[0, 0]
cols1 = [PALETTE[i % len(PALETTE)] for i in range(len(tipo_sum))]
bars1 = ax1.bar(tipo_sum.index, tipo_sum.values, color=cols1, edgecolor='white', linewidth=0.8)
off1 = tipo_sum.max() * 0.015
for bar, v in zip(bars1, tipo_sum.values):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + off1,
             f'{int(v):,}\n({v/total_sum_tipo*100:.1f}%)',
             ha='center', va='bottom', fontsize=10, fontweight='bold', color=C_VERDE)
ax1.set_title(f'Suma de mascotas por tipo  |  {int(total_sum_tipo):,} total', fontweight='bold')
ax1.set_ylabel('N° mascotas')

# Panel 2: Distribución de cantidad de mascotas por contrato
ax2 = axes[0, 1]
_cant_clip = masc_det['Cantidad_masc'].clip(upper=5).replace(5, '5+').astype(str)
cant_cnt = (
    _cant_clip.value_counts()
    .reindex([str(i) for i in range(1, 5)] + ['5+'], fill_value=0)
)
cols2 = [PALETTE[i % len(PALETTE)] for i in range(len(cant_cnt))]
bars2 = ax2.bar(cant_cnt.index, cant_cnt.values, color=cols2, edgecolor='white', linewidth=0.8)
off2 = cant_cnt.max() * 0.015
for bar, v in zip(bars2, cant_cnt.values):
    if v > 0:
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + off2,
                 f'{v:,}\n({v/total_contratos_masc*100:.1f}%)',
                 ha='center', va='bottom', fontsize=10, fontweight='bold', color=C_VERDE)
ax2.set_title('Cantidad de mascotas por contrato', fontweight='bold')
ax2.set_xlabel('N° mascotas')
ax2.set_ylabel('Contratos')

# Panel 3: Edades individuales
ax3 = axes[1, 0]
if len(todas_edades) > 0:
    ax3.hist(todas_edades, bins=25, color=PALETTE[0], edgecolor='white', linewidth=0.6)
    ax3.axvline(todas_edades.median(), color=C_AMARILLO, linewidth=2,
                linestyle='--', label=f'Mediana: {todas_edades.median():.1f} años')
    ax3.axvline(todas_edades.mean(), color=C_V_PAL, linewidth=2,
                linestyle='--', label=f'Media: {todas_edades.mean():.1f} años')
    ax3.legend(fontsize=9)
    ax3.set_title(f'Edad de mascotas — {total_edades:,} válidas (≤{EDAD_MAX_MASCOTA} años)', fontweight='bold')
    ax3.set_xlabel('Edad (años)')
    ax3.set_ylabel('Mascotas')
else:
    ax3.text(0.5, 0.5, 'Sin datos de edad', ha='center', va='center',
             transform=ax3.transAxes, fontsize=12)
    ax3.set_title('Edad de mascotas', fontweight='bold')

# Panel 4: Top razas
ax4 = axes[1, 1]
if _col_razas:
    razas_serie = (
        masc_det['Razas'].dropna().astype(str).str.strip()
        .replace({'nan': '', 'None': '', 'none': ''})
    )
    razas_serie = razas_serie[razas_serie != '']
    if len(razas_serie) > 0:
        top_razas = razas_serie.value_counts().head(10)
        cols4 = [PALETTE[i % len(PALETTE)] for i in range(len(top_razas))]
        bars4 = ax4.barh(top_razas.index[::-1], top_razas.values[::-1],
                         color=cols4[::-1], edgecolor='white', linewidth=0.6)
        off4 = top_razas.max() * 0.01
        for bar, v in zip(bars4, top_razas.values[::-1]):
            ax4.text(bar.get_width() + off4, bar.get_y() + bar.get_height() / 2,
                     f'{v:,}', va='center', fontsize=9, fontweight='bold', color=C_VERDE)
        ax4.set_title('Top 10 razas', fontweight='bold')
        ax4.set_xlabel('Contratos')
        ax4.set_xlim(0, top_razas.max() * 1.18)
    else:
        ax4.text(0.5, 0.5, 'Sin datos de razas', ha='center', va='center',
                 transform=ax4.transAxes, fontsize=12)
        ax4.set_title('Top razas', fontweight='bold')

plt.suptitle(f'Mascotas — Análisis detallado  |  {total_contratos_masc:,} contratos  |  {int(total_sum_tipo):,} mascotas',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Top 10 contratos con más mascotas ─────────────────────────────────────────
print(f"\n{'─'*70}")
print(f"  TOP 10 contratos con más mascotas")
print(f"{'─'*70}")
top10 = (
    masc_det[['Contrato', 'Cantidad_masc', 'Tipo_mascota', 'Edades_raw', 'Razas']]
    .sort_values('Cantidad_masc', ascending=False)
    .head(10)
    .reset_index(drop=True)
)
top10.index += 1
print(top10.to_string())

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PÓLIZA — Contratos y breakdown por tipo (AP / PFI / SOLICANASTA)
# ══════════════════════════════════════════════════════════════════════════════

total_contratos_base = len(df_tipo)
con_poliza = int(df_tipo['Poliza'].sum())
sin_poliza = total_contratos_base - con_poliza

poliza_tipo = (
    df_modelo[df_modelo['tiene_poliza']]
    .groupby('Contrato')['Tiposseguros_ajuste']
    .first()
    .value_counts()
    .reset_index()
    .rename(columns={'Tiposseguros_ajuste': 'Tipo', 'count': 'Contratos'})
    .sort_values('Contratos', ascending=False)
    .reset_index(drop=True)
)
print(f"Contratos CON seguro : {con_poliza:,}  ({con_poliza/total_contratos_base*100:.1f}%)")
print(f"Contratos SIN seguro : {sin_poliza:,}  ({sin_poliza/total_contratos_base*100:.1f}%)")
print("\nBreakdown por tipo:")
for _, row in poliza_tipo.iterrows():
    print(f"  {row['Tipo']:<15}  {int(row['Contratos']):>8,}   ({row['Contratos']/con_poliza*100:.1f}% de contratos con seguro)")

# ── Gráfico ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel izq — Con / Sin Seguro
ax1 = axes[0]
vals1 = [con_poliza, sin_poliza]
bars1 = ax1.bar(['Con Seguro', 'Sin Seguro'], vals1,
                color=[PALETTE[0], PALETTE[3]], edgecolor='white', linewidth=0.8, width=0.5)
ax1.set_ylim(0, max(vals1) * 1.22)
off1 = max(vals1) * 0.012
for bar, v in zip(bars1, vals1):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + off1,
             f'{v:,}\n({v/total_contratos_base*100:.1f}%)',
             ha='center', va='bottom', fontsize=9, fontweight='bold', color=C_VERDE)
ax1.set_title('Seguro— Contratos', fontweight='bold')
ax1.set_ylabel('Contratos únicos')

# Panel der — Breakdown por tipo
ax2 = axes[1]
cols2 = [PALETTE[i % len(PALETTE)] for i in range(len(poliza_tipo))]
bars2 = ax2.bar(poliza_tipo['Tipo'], poliza_tipo['Contratos'],
                color=cols2, edgecolor='white', linewidth=0.8, width=0.5)
ax2.set_ylim(0, poliza_tipo['Contratos'].max() * 1.22)
off2 = poliza_tipo['Contratos'].max() * 0.012
for bar, v in zip(bars2, poliza_tipo['Contratos']):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + off2,
             f'{int(v):,}\n({v/con_poliza*100:.1f}%)',
             ha='center', va='bottom', fontsize=9, fontweight='bold', color=C_VERDE)
ax2.set_title('Seguro — Tipo (AP / PFI / SOLICANASTA)', fontweight='bold')
ax2.set_ylabel('Contratos únicos')

plt.suptitle(f'Seguro  |  {con_poliza:,} contratos con Seguro de {total_contratos_base:,} totales',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# COBERTURAS ADICIONALES — Salud / Bicicleta / Repatriacion / Expatriacion
# ══════════════════════════════════════════════════════════════════════════════

coberturas = ['Salud', 'Bicicleta', 'Repatriacion', 'Expatriacion']
total_base  = len(df_tipo)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, col in zip(axes, coberturas):
    con = int(df_tipo[col].sum())
    sin = total_base - con
    vals = [con, sin]
    etqs = [f'Con {col}', f'Sin {col}']
    pcts = [v / total_base * 100 for v in vals]

    print(f"{col:<14} CON: {con:>7,} ({pcts[0]:.1f}%)   SIN: {sin:>7,} ({pcts[1]:.1f}%)")

    bars = ax.bar(etqs, vals,
                  color=[PALETTE[0], PALETTE[3]],
                  edgecolor='white', linewidth=0.8, width=0.5)
    ax.set_ylim(0, max(vals) * 1.22)
    off = max(vals) * 0.012
    for bar, v, p in zip(bars, vals, pcts):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + off,
                f'{v:,}\n({p:.1f}%)',
                ha='center', va='bottom',
                fontsize=9, fontweight='bold', color=C_VERDE)
    ax.set_title(col, fontweight='bold')
    ax.set_ylabel('Contratos únicos')
    ax.tick_params(axis='x', labelsize=9)

plt.suptitle(f'Coberturas adicionales  |  {total_base:,} contratos totales',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Validación QLIK

In [ ]:
import re

# ─────────────────────────────────────────────────────────────────────────────
# 1. DETECTAR NOMBRE EXACTO DE LA COLUMNA EN PAP_clientes
# ─────────────────────────────────────────────────────────────────────────────
col_edades = next(
    (c for c in PAP_clientes.columns if c.lower() == 'edadesmascotas'), None
)
col_estado = next(
    (c for c in PAP_clientes.columns if c.lower() == 'estado'), None
)
if col_edades is None:
    raise KeyError(f"No se encontró EdadesMascotas. Columnas: {list(PAP_clientes.columns)}")
if col_estado is None:
    raise KeyError(f"No se encontró Estado. Columnas: {list(PAP_clientes.columns)}")
print(f"Columnas detectadas: '{col_edades}' | '{col_estado}'")

# ─────────────────────────────────────────────────────────────────────────────
# 2. FUNCIÓN BASE (replica SubStringCount de Qlik)
# ─────────────────────────────────────────────────────────────────────────────
def contar_mascotas_qlik(val):
    if pd.isna(val):
        return 0
    v = str(val).strip()
    if v == '' or v.lower() == 'nan':
        return 0
    v_clean = re.sub(r'-{2,}', '-', v).strip('-')
    if v_clean == '':
        return 0
    return v_clean.count('-') + 1

# ─────────────────────────────────────────────────────────────────────────────
# 3. CON DUPLICADOS (como Qlik) vs SIN DUPLICADOS
# ─────────────────────────────────────────────────────────────────────────────
df_diag = PAP_clientes[['Contrato', col_estado, col_edades]].copy()
df_diag['Cantidad_mascotas'] = df_diag[col_edades].apply(contar_mascotas_qlik)

total_con_dup    = df_diag['Cantidad_mascotas'].sum()
n_filas          = len(df_diag)
n_contratos_uniq = df_diag['Contrato'].nunique()
n_duplicados     = n_filas - n_contratos_uniq

df_uniq       = df_diag.groupby('Contrato')['Cantidad_mascotas'].max().reset_index()
total_sin_dup = df_uniq['Cantidad_mascotas'].sum()

diff_abs = total_con_dup - total_sin_dup
diff_pct = diff_abs / total_con_dup * 100

# ─────────────────────────────────────────────────────────────────────────────
# 4. TABLA COMPARATIVA GLOBAL
# ─────────────────────────────────────────────────────────────────────────────
comparativo = pd.DataFrame({
    'Métrica' : ['Filas totales', 'Contratos únicos', 'Filas duplicadas',
                 'sum() CON duplicados (Qlik)', 'sum() SIN duplicados (Python dedup)',
                 'Diferencia absoluta', 'Diferencia %'],
    'Valor'   : [f'{n_filas:,}', f'{n_contratos_uniq:,}', f'{n_duplicados:,}',
                 f'{int(total_con_dup):,}', f'{int(total_sin_dup):,}',
                 f'{int(diff_abs):,}', f'{diff_pct:.2f}%'],
})
print("\n" + "=" * 55)
print("TABLA COMPARATIVA GLOBAL — Cantidad_mascotas")
print("=" * 55)
print(comparativo.to_string(index=False))

# ─────────────────────────────────────────────────────────────────────────────
# 5. VALIDACIÓN POR ESTADO
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("VALIDACIÓN POR ESTADO")
print("=" * 55)

# Con duplicados (Qlik)
por_estado_dup = (
    df_diag.groupby(col_estado)['Cantidad_mascotas']
    .agg(contratos_filas='count', sum_con_dup='sum')
    .reset_index()
    .rename(columns={col_estado: 'Estado'})
)

# Sin duplicados — primero max por (Contrato, Estado), luego sum
df_uniq_estado = (
    df_diag.groupby(['Contrato', col_estado])['Cantidad_mascotas']
    .max()
    .reset_index()
    .rename(columns={col_estado: 'Estado'})
)
por_estado_uniq = (
    df_uniq_estado.groupby('Estado')
    .agg(contratos_uniq=('Contrato','nunique'),
         sum_sin_dup=('Cantidad_mascotas','sum'))
    .reset_index()
)

resumen_estado = (
    por_estado_dup
    .merge(por_estado_uniq, on='Estado', how='outer')
    .fillna(0)
)
resumen_estado['diferencia']  = (resumen_estado['sum_con_dup'] - resumen_estado['sum_sin_dup']).astype(int)
resumen_estado['diff_pct']    = (resumen_estado['diferencia'] / resumen_estado['sum_con_dup'].replace(0, np.nan) * 100).round(2)
resumen_estado['sum_con_dup'] = resumen_estado['sum_con_dup'].astype(int)
resumen_estado['sum_sin_dup'] = resumen_estado['sum_sin_dup'].astype(int)

print(resumen_estado[['Estado','contratos_filas','contratos_uniq',
                       'sum_con_dup','sum_sin_dup','diferencia','diff_pct']]
      .to_string(index=False))

# ─────────────────────────────────────────────────────────────────────────────
# 6. DIAGNÓSTICO DE CAUSAS
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("DIAGNÓSTICO DE DISCREPANCIAS")
print("=" * 55)

dup_contratos = (
    df_diag.groupby('Contrato')
    .agg(n_filas=('Cantidad_mascotas','count'),
         sum_filas=('Cantidad_mascotas','sum'),
         max_val=('Cantidad_mascotas','max'))
    .query('n_filas > 1')
    .reset_index()
)
dup_contratos['exceso'] = dup_contratos['sum_filas'] - dup_contratos['max_val']
exceso_total = dup_contratos['exceso'].sum()

print(f"\n[1] Contratos duplicados          : {len(dup_contratos):,}")
print(f"    Mascotas extra por duplicados  : {int(exceso_total):,}")
if len(dup_contratos) > 0:
    print(f"    Top 5 contratos más duplicados:")
    print(dup_contratos.nlargest(5, 'n_filas')
          [['Contrato','n_filas','sum_filas','max_val','exceso']]
          .to_string(index=False))

n_nulos   = df_diag[col_edades].isna().sum()
n_vacios  = (df_diag[col_edades].astype(str).str.strip() == '').sum()
n_nan_str = (df_diag[col_edades].astype(str).str.lower().str.strip() == 'nan').sum()
print(f"\n[2] Nulos (NaN)        : {n_nulos:,}")
print(f"    Strings vacíos     : {n_vacios:,}")
print(f"    String 'nan'       : {n_nan_str:,}")

mask_doble = df_diag[col_edades].astype(str).str.contains(r'-{2,}', na=False)
mask_borde = df_diag[col_edades].astype(str).str.strip().str.match(r'^-|-$', na=False)
mask_solo  = df_diag[col_edades].astype(str).str.strip().str.match(r'^-+$', na=False)
print(f"\n[3] Formatos inconsistentes:")
print(f"    Doble guion ('2--5') : {mask_doble.sum():,}")
print(f"    Guion al borde ('-3'): {mask_borde.sum():,}")
print(f"    Solo guiones ('---') : {mask_solo.sum():,}")
if mask_doble.sum() > 0:
    print("    Ejemplos:")
    print(df_diag.loc[mask_doble, col_edades].value_counts().head(5).to_string())

# ─────────────────────────────────────────────────────────────────────────────
# 7. GRÁFICO
# ─────────────────────────────────────────────────────────────────────────────
n_estados = len(resumen_estado)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1 — Global con vs sin duplicados
ax1 = axes[0]
etiquetas = ['Qlik\n(con dup.)', 'Python\n(sin dup.)']
valores   = [int(total_con_dup), int(total_sin_dup)]
bars = ax1.bar(etiquetas, valores, color=[PALETTE[2], PALETTE[0]],
               edgecolor='white', linewidth=0.8, width=0.5)
offset = max(valores) * 0.015
for bar, v in zip(bars, valores):
    ax1.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + offset,
             f'{v:,}', ha='center', va='bottom',
             fontsize=11, fontweight='bold', color=C_VERDE)
ax1.set_title('sum(Cantidad_mascotas)\nGlobal', fontweight='bold')
ax1.set_ylabel('Total mascotas')

# Panel 2 — sum por Estado (con vs sin dup)
ax2 = axes[1]
x      = np.arange(n_estados)
width  = 0.35
bars_a = ax2.bar(x - width/2, resumen_estado['sum_con_dup'], width,
                 label='Con dup. (Qlik)', color=PALETTE[2], edgecolor='white')
bars_b = ax2.bar(x + width/2, resumen_estado['sum_sin_dup'], width,
                 label='Sin dup. (Python)', color=PALETTE[0], edgecolor='white')
for bar in list(bars_a) + list(bars_b):
    v = bar.get_height()
    if v > 0:
        ax2.text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + resumen_estado['sum_con_dup'].max() * 0.01,
                 f'{int(v):,}', ha='center', va='bottom',
                 fontsize=8, fontweight='bold', color=C_VERDE)
ax2.set_xticks(x)
ax2.set_xticklabels(resumen_estado['Estado'], rotation=20, ha='right')
ax2.set_title('sum(Cantidad_mascotas)\npor Estado', fontweight='bold')
ax2.set_ylabel('Total mascotas')
ax2.legend()

# Panel 3 — Diferencia por Estado
ax3 = axes[2]
colores3 = [PALETTE[i % len(PALETTE)] for i in range(n_estados)]
bars3 = ax3.bar(resumen_estado['Estado'], resumen_estado['diferencia'],
                color=colores3, edgecolor='white', linewidth=0.8)
offset3 = resumen_estado['diferencia'].max() * 0.015
for bar, v, pct in zip(bars3, resumen_estado['diferencia'], resumen_estado['diff_pct']):
    if v > 0:
        ax3.text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + offset3,
                 f'{int(v):,}\n({pct:.1f}%)', ha='center', va='bottom',
                 fontsize=8, fontweight='bold', color=C_VERDE)
ax3.set_title('Diferencia (dup - dedup)\npor Estado', fontweight='bold')
ax3.set_ylabel('Mascotas extra por duplicados')
ax3.tick_params(axis='x', rotation=20)

plt.suptitle('Análisis Cantidad_mascotas — Qlik vs Python por Estado',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# 8. CONCLUSIÓN
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("CONCLUSIÓN")
print("=" * 55)
print(f"  Qlik sum()            : {int(total_con_dup):,}")
print(f"  Python sin duplicados : {int(total_sin_dup):,}")
print(f"  Diferencia            : {int(diff_abs):,}  ({diff_pct:.2f}%)")
print(f"\n  Causa principal: {int(exceso_total):,} mascotas extra por")
print(f"  {len(dup_contratos):,} contratos duplicados en PAP_clientes.")
print(f"\n  Estado con mayor discrepancia:")
idx_max = resumen_estado['diferencia'].idxmax()
row_max = resumen_estado.loc[idx_max]
print(f"  → {row_max['Estado']}: {int(row_max['diferencia']):,} mascotas extra ({row_max['diff_pct']:.1f}%)")
print(f"\n  Valor correcto para análisis: {int(total_sin_dup):,} (sin duplicados)")